In [ ]:
# Data root is configurable: export SYNERGPCR_BASE=/path/to/released/tables
# (defaults to ./data). All input paths below are resolved against it.
from pathlib import Path
import os
import pandas as pd
import time
import json
from tqdm import tqdm
import google.generativeai as genai
from typing_extensions import TypedDict

BASE = Path(os.environ.get("SYNERGPCR_BASE", "./data"))


# ==========================================
# 1. API Configuration
# ==========================================
API_KEY = os.environ.get("GEMINI_API_KEY")  # Insert your API key here
genai.configure(api_key=API_KEY)

class MoAExtraction(TypedDict):
    moa_label: str  

model = genai.GenerativeModel(
    'models/gemini-2.5-flash-lite',
    generation_config={
        "response_mime_type": "application/json",
        "response_schema": MoAExtraction,
        "temperature": 0.0  
    }
)

In [ ]:
# ==========================================
# 2. LLM Extraction Function for GLASS v2.0
# ==========================================
def extract_glass_moa_llm(standard_type, classification, action_type, retries=7):
    """
    Calls the Gemini API to classify the MoA using GLASS v2.0 specific columns.
    """
    time.sleep(1) 
    
    prompt = f"""
    You are an expert pharmacologist mapping GPCR ligand data.
    Determine the final Mechanism of Action (MoA) based strictly on the provided assay parameters from the GLASS database.
    
    CRITICAL RULES:
    1. "EC50" and "active" do NOT automatically mean "agonist". They only indicate activity in a functional assay.
    2. If the 'Action Type' is ambiguous (e.g., "MODULATION", "OTHER") and doesn't clearly state activation or inhibition, you MUST output "binder".
    3. Only output "agonist" or "pam" if the Action Type clearly implies positive efficacy (e.g., UPREGULATION).
    4. Only output "antagonist" or "nam" if the Action Type implies negative/blocking efficacy (e.g., DOWNREGULATION).
    5. Do not over-interpret. When in doubt, ALWAYS default to "binder".
    
    Classify into exactly one of the following labels:
    - "agonist" 
    - "antagonist"
    - "partial agonist"
    - "inverse agonist"
    - "pam" 
    - "nam" 
    - "binder" 
    
    Return ONLY the matching label in lowercase. Do not use quotes, code blocks, or extra text.
    
    Standard Type: "{standard_type}"
    Classification: "{classification}"
    Action Type: "{action_type}"
    """
    
    valid_labels = ["inverse agonist", "partial agonist", "agonist", "antagonist", "pam", "nam", "binder"]
    wait = 5
    
    for attempt in range(retries):
        try:
            response = model.generate_content(prompt)
            raw_text = response.text.lower().strip()
            
            for valid_label in valid_labels:
                if valid_label in raw_text:
                    return valid_label
                    
            return "binder"
            
        except Exception as e:
            if "429" in str(e) or "ResourceExhausted" in str(e):
                print(f"Rate limit. Waiting {wait}s... (attempt {attempt+1}/{retries})")
                time.sleep(wait)
                wait *= 2
            else:
                print(f"API Error: {e}")
                break
                
    return "binder"

In [ ]:
# ==========================================
# 3. Hybrid Processing Pipeline for GLASS
# ==========================================
def process_glass_moa_hybrid(tsv_path, output_dir, valid_uniprot_list):
    print("\n--- Processing GLASS v2.0 MoA (Hybrid Rule + LLM) ---")
    
    # Read TSV file
    df = pd.read_csv(tsv_path, sep='\t', low_memory=False)
    
    # Filter only essential columns
    essential_cols = [
        'compound_inchikey', 'target_uniprot_id', 'source_database', 
        'standard_type', 'standard_value', 'standard_units', 
        'classification', 'action_type'
    ]
    df = df[essential_cols].copy()
    
    # Drop rows where critical ligand or target information is missing
    df.dropna(subset=['compound_inchikey', 'target_uniprot_id'], inplace=True)
    
    # --- NEW: Filter by Human GPCR UniProt IDs ---
    initial_row_count = len(df)
    df = df[df['target_uniprot_id'].isin(valid_uniprot_list)].copy()
    print(f"Filtered for Human GPCRs: from {initial_row_count} to {len(df)} rows.") 
    
    def determine_path(row):
        action = str(row['action_type']).lower().strip()
        
        # 'nan', 'none', 빈 문자열('')을 명확한 Rule 대상으로 추가
        clear_actions = ['activation', 'inhibition', 'binding', 'nan', 'none', '']
        
        # If it's an exact match to a clear action or missing, use Rule
        if action in clear_actions:
            return 'Rule'
        
        # Ambiguous terms (modulation, upregulation, other) go to LLM
        return 'LLM'
        
    df['Processing_Path'] = df.apply(determine_path, axis=1)
    
    # Apply Rule-based Mapping
    def apply_rule(action):
        action = str(action).lower().strip()
        
        # Map GLASS action types to your standard MoA labels
        if action == 'activation': 
            return 'agonist'
        elif action == 'inhibition': 
            return 'antagonist'
        elif action in ['binding', 'nan', 'none', '']: 
            # action 정보가 없거나 단순히 결합만 확인된 경우 모두 binder로 처리
            return 'binder'
            
        return 'binder' # Fallback
        
    rule_mask = df['Processing_Path'] == 'Rule'
    df.loc[rule_mask, 'MoA_Label'] = df.loc[rule_mask, 'action_type'].apply(apply_rule)
    df.loc[rule_mask, 'Label_Source'] = 'Rule'
    
    # Apply LLM-based Mapping
    llm_mask = df['Processing_Path'] == 'LLM'
    llm_target_df = df[llm_mask].copy()
    
    # Extract unique combinations to minimize API calls
    unique_pairs = llm_target_df[['standard_type', 'classification', 'action_type']].drop_duplicates()
    unique_pairs.fillna('', inplace=True)
    
    print(f"Total rows: {len(df)} | Rule-based: {rule_mask.sum()} | LLM targets (Unique): {len(unique_pairs)}")
    
    CHECKPOINT_FILE = os.path.join(output_dir, "glass_moa_checkpoint.json")
    mapping_results = {}
    
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            mapping_results = json.load(f)
        print(f"Resumed from checkpoint: {len(mapping_results)} combinations already done.")

    # Process unique combinations via LLM
    for i, row in tqdm(unique_pairs.iterrows(), total=len(unique_pairs), desc='Mining MoA via LLM'):
        std_type = str(row['standard_type'])
        cls_val = str(row['classification'])
        act_val = str(row['action_type'])
        
        pair_key = f"{std_type} ||| {cls_val} ||| {act_val}"
        
        if pair_key in mapping_results:
            continue
            
        label = extract_glass_moa_llm(std_type, cls_val, act_val)
        mapping_results[pair_key] = label
        
        # Checkpoint save
        if (i + 1) % 50 == 0:
            with open(CHECKPOINT_FILE, "w") as f:
                json.dump(mapping_results, f, ensure_ascii=False)

    # Final checkpoint save
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(mapping_results, f, ensure_ascii=False)
        
    # Map results back to dataframe
    def map_llm_result(row):
        std_type = str(row['standard_type']).replace('nan', '')
        cls_val = str(row['classification']).replace('nan', '')
        act_val = str(row['action_type']).replace('nan', '')
        pair_key = f"{std_type} ||| {cls_val} ||| {act_val}"
        return mapping_results.get(pair_key, 'binder')

    df.loc[llm_mask, 'MoA_Label'] = df[llm_mask].apply(map_llm_result, axis=1)
    df.loc[llm_mask, 'Label_Source'] = 'LLM'
    
    # Final Cleanup and Renaming
    df.drop(columns=['Processing_Path'], inplace=True)
    df.rename(columns={'target_uniprot_id': 'UniProt_AC'}, inplace=True)
    
    # Save Final Data
    out_path = os.path.join(output_dir, 'GLASS_v2_MoA_Labeled.csv')
    df.to_csv(out_path, index=False)
    
    print("\n=== Final GLASS MoA Distribution ===")
    print(df['MoA_Label'].value_counts())
    print(f"\nSaved standardized GLASS MoA to: {out_path}")

In [ ]:
if __name__ == "__main__":
    # --- A. Load Human GPCR Target List ---
    human_gpcr_df = pd.read_csv('./Input/Human_GPCR_PDB_Info.csv')
    human_uniprot_list = human_gpcr_df['Entry'].dropna().unique().tolist()
    print(f"Loaded {len(human_uniprot_list)} Human GPCR UniProt IDs.")
    
    input_tsv = str(BASE / "DB/GLASS/glass2_full.tsv")
    output_directory = str(BASE / "Output/DB/GLASS/NAR/")
    os.makedirs(output_directory, exist_ok=True)

    process_glass_moa_hybrid(input_tsv, output_directory, human_uniprot_list)

In [ ]:
tmp = pd.read_csv(str(BASE / "Output/DB/GLASS/NAR/GLASS_v2_MoA_Labeled.csv"))

In [ ]:
tmp